# Day 35 — Practice ANSWERS
### Yes or No — logistic regression

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

| Part | What it asks you to do | How many |
|---|---|---|
| **A** | Work it out — pen and paper | 8 |
| **B** | Find the mistake in a short program | 4 |
| **C** | Write the code yourself | 5 |
| **D** | Challenge | 2 |

### How to work

**Part A needs a calculator at most.** Work it out, then run the cell to check.

Three questions to keep asking:

- Is this coefficient a probability, or a **log-odds**? (Always the second.)
- What is the **baseline**? Accuracy means nothing without it.
- Who chose the **cut-off**, and what does each mistake cost?

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, log_loss

import warnings
warnings.simplefilter("ignore")
plt.rcParams["figure.figsize"] = (8, 3.2)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

print("ready")

ready


---
# Part A — Work it out

A bank fits this model to predict whether a loan will be **repaid**:

```
z  =  0.42 × income_lakhs  −  0.65 × existing_loans  +  0.018 × credit_score  −  12.0
p  =  1 / (1 + e⁻ᶻ)
```

### A1 · The sigmoid, by hand

Work out `sigmoid(z)` for z = −2, 0 and +2. Do z = 0 without a calculator.

In [2]:
for z in [-2, 0, 2]:
    print(f"   sigmoid({z:>2}) = {sigmoid(z):.4f}")

   sigmoid(-2) = 0.1192
   sigmoid( 0) = 0.5000
   sigmoid( 2) = 0.8808


| z | p |
|---|---|
| −2 | **0.1192** |
| 0 | **0.5000** |
| +2 | **0.8808** |

**z = 0 needs no calculator:** e⁰ = 1, so p = 1/(1+1) = **exactly 0.5**.

And notice the symmetry: sigmoid(−2) = 0.1192 and sigmoid(+2) = 0.8808, which add to
exactly 1. The curve is symmetric about (0, 0.5) — so sigmoid(−z) is always 1 − sigmoid(z).

### A2 · Score one applicant

Income **8 lakhs**, **1** existing loan, credit score **720**.

Work out z, then p. Would the bank approve at a 0.5 cut-off?

In [3]:
def z_of(income, loans, score):
    return 0.42*income - 0.65*loans + 0.018*score - 12.0

z = z_of(8, 1, 720)
print(f"z = {z:.4f}")
print(f"p = {sigmoid(z):.4f}")

z = 3.6700
p = 0.9752


```
z = 0.42 × 8  −  0.65 × 1  +  0.018 × 720  −  12.0
  = 3.36      −  0.65      +  12.96        −  12.0
  = 3.67
p = 1 / (1 + e^-3.67) = 0.9752
```

**p = 0.975, so yes — approve.** And note the useful part: you can tell this applicant
"our model puts your repayment chance at about 97.5%", not just "approved".

### A3 · What does +0.42 mean?

A colleague says *"income adds 0.42 to the probability of repaying."*

Explain why that is wrong, and write the correct sentence.

In [4]:
print("one extra lakh of income multiplies the odds by:", round(np.exp(0.42), 4))
print()
for inc in [5, 6, 7, 8]:
    p = sigmoid(z_of(inc, 1, 720))
    print(f"   income {inc} lakhs: p = {p:.4f},  odds = {p/(1-p):.4f}")

one extra lakh of income multiplies the odds by: 1.522

   income 5 lakhs: p = 0.9176,  odds = 11.1340
   income 6 lakhs: p = 0.9443,  odds = 16.9455
   income 7 lakhs: p = 0.9627,  odds = 25.7903
   income 8 lakhs: p = 0.9752,  odds = 39.2519


**Why it is wrong:** probabilities stop at 1. If one lakh added 0.42, then three extra
lakhs would take an applicant past certainty — which is meaningless.

Look at what actually happens:

```
one extra lakh of income multiplies the odds by: 1.522

   income 5 lakhs: p = 0.9176,  odds = 11.1340
   income 6 lakhs: p = 0.9443,  odds = 16.9455
   income 7 lakhs: p = 0.9627,  odds = 25.7903
   income 8 lakhs: p = 0.9752,  odds = 39.2519
```

The probability steps are **+0.0267, +0.0184, +0.0125** — shrinking every time, because the
curve is flattening out as it approaches 1.

But work out the odds ratios:

```
16.9455 / 11.1340 = 1.522
25.7903 / 16.9455 = 1.522
39.2519 / 25.7903 = 1.522
```

**Identical, every single step.** That is the thing the coefficient is constant in.

**The correct sentence:**

> "Holding existing loans and credit score constant, each extra lakh of income multiplies
> the **odds** of repaying by **e⁰·⁴² = 1.522** — about a 52% increase in the odds."

### A4 · Which column matters most?

The three coefficients are **+0.42**, **−0.65** and **+0.018**.

A colleague concludes that `credit_score` barely matters because 0.018 is tiny. Is he
right?

**No — and this is the Day 34 trap, repeated.**

A coefficient is **per unit of its own column**, and these three columns have wildly
different ranges:

| column | realistic range | the model's total swing across it |
|---|---|---|
| `income_lakhs` | 2 to 30 | 0.42 × 28 ≈ **11.8** in z |
| `existing_loans` | 0 to 4 | 0.65 × 4 = **2.6** |
| `credit_score` | 300 to 850 | 0.018 × 550 ≈ **9.9** |

`credit_score` has by far the **smallest** coefficient and the **second largest**
influence, because a credit score moves in hundreds while income moves in single digits.

**The fix:** scale every column before comparing, so each coefficient means "per standard
deviation". Or do the rough version above — coefficient × the column's realistic spread.

### A5 · Odds and probability

Fill in the gaps.

| probability | odds |
|---|---|
| 0.5 | ? |
| 0.8 | ? |
| ? | 9 |
| 0.25 | ? |

In [5]:
for p in [0.5, 0.8, 0.25]:
    print(f"   p = {p}  ->  odds = {p/(1-p):.4f}")
print()
for o in [9]:
    print(f"   odds = {o}  ->  p = {o/(1+o):.4f}")

   p = 0.5  ->  odds = 1.0000
   p = 0.8  ->  odds = 4.0000
   p = 0.25  ->  odds = 0.3333

   odds = 9  ->  p = 0.9000


| probability | odds |
|---|---|
| 0.5 | **1** |
| 0.8 | **4** |
| **0.9** | 9 |
| 0.25 | **0.333** |

Two conversions worth memorising:

```
odds = p / (1 - p)          p = odds / (1 + odds)
```

**Odds of 1 means a 50/50 chance** — "evens", as a bookmaker would say. Odds above 1 mean
more likely than not; below 1, less likely than not.

### A6 · Log loss by hand

A model gives these four probabilities. Work out the log loss for each row, then the
average.

| | true answer | p the model gave |
|---|---|---|
| 1 | yes | 0.9 |
| 2 | yes | 0.6 |
| 3 | no | 0.2 |
| 4 | no | 0.8 |

In [6]:
truth = np.array([1., 1, 0, 0])
p = np.array([0.9, 0.6, 0.2, 0.8])

cost = np.where(truth == 1, -np.log(p), -np.log(1 - p))
for t, pp, c in zip(truth, p, cost):
    print(f"   truth {int(t)},  p = {pp}  ->  cost {c:.4f}")
print(f"\n   average log loss: {cost.mean():.4f}")
print(f"   sklearn         : {log_loss(truth, p):.4f}")

   truth 1,  p = 0.9  ->  cost 0.1054
   truth 1,  p = 0.6  ->  cost 0.5108
   truth 0,  p = 0.2  ->  cost 0.2231
   truth 0,  p = 0.8  ->  cost 1.6094

   average log loss: 0.6122
   sklearn         : 0.6122


| | truth | p | which formula | cost |
|---|---|---|---|---|
| 1 | yes | 0.9 | −ln(0.9) | **0.1054** |
| 2 | yes | 0.6 | −ln(0.6) | **0.5108** |
| 3 | no | 0.2 | −ln(1 − 0.2) = −ln(0.8) | **0.2231** |
| 4 | no | 0.8 | −ln(1 − 0.8) = −ln(0.2) | **1.6094** |

**Average = 0.6122**

**Row 4 is the story.** The model said 0.8 for something that did not happen, and that one
row costs **1.61** — more than the other three put together.

That is the cost function doing its job: a confident mistake is punished hard. Under
squared error row 4 would have cost 0.64, barely three times row 2's 0.16.

### A7 · Is 97% accuracy good?

A model predicts whether an email is phishing. **2 emails in 100 are phishing.**
The model is **97% accurate**.

What is the most likely explanation, and what should you ask for instead?

**The most likely explanation: the model says "not phishing" to almost everything.**

A model that *never* flags anything is **98% accurate** on this data, with no model at
all. So 97% is not just unimpressive — it is **worse than doing nothing**.

**What to ask for instead:**

1. **The baseline.** "Always say no" scores 98%. Quote accuracy next to it or not at all.
2. **The confusion matrix.** Of the 2 phishing emails in every 100, how many did it catch?
3. **Recall on the rare class.** Catching 1 of 2 is 50% recall, and that is the number
   that decides whether the thing is useful.

**The general rule:** when one class is rare, accuracy is the wrong question. It is
dominated by the easy majority. This is the Imbalanced Data day later in the module.

### A8 · Choosing a cut-off

Two models, both at a 0.5 cut-off:

| | what it does | a false YES costs | a false NO costs |
|---|---|---|---|
| **A** | flags a tumour on a scan for a specialist to review | one unnecessary review, ~₹2,000 | a missed cancer |
| **B** | auto-blocks a credit card transaction as fraud | a customer stranded abroad | the bank refunds the fraud |

For each, should you move the cut-off up or down from 0.5?

**Model A — move the cut-off DOWN**, perhaps a long way, to 0.2 or lower.

A false yes costs ₹2,000 and a specialist's afternoon. A false no may cost a life. Those
are not remotely comparable, so you should accept a great many false alarms to miss as
few real cases as possible. **Lowering the cut-off raises recall** — you flag more, and
miss fewer.

**Model B — move the cut-off UP**, to 0.8 or higher.

A false yes strands a real customer with a declined card, possibly in another country.
A false no costs the bank money it can reclaim. The bank can absorb its own mistake; the
customer cannot absorb theirs. **Raising the cut-off raises precision** — you block less,
and you are right more often when you do.

### The principle

**0.5 is not a default that happens to be right. It is the answer you get when both
mistakes cost the same — and they almost never do.**

Write down what each mistake costs *before* you pick a number. If you cannot say which
mistake is worse, you do not yet understand the problem well enough to deploy a model.

---
# Part B — Find the mistake

Four short programs. Each runs cleanly and reports something misleading.

In [7]:
# ---------------------------------------------------------------
# 900 patients. B1 to B4 all use `scans`.
# ---------------------------------------------------------------
rng = np.random.default_rng(3540)
n = 900

age       = rng.normal(55, 14, n).clip(20, 90).round(0)
lump_mm   = rng.gamma(2.0, 3.2, n).clip(0, 40).round(1)
family    = rng.binomial(1, 0.22, n).astype(float)

z = -6.3 + 0.035*age + 0.27*lump_mm + 0.70*family + rng.normal(0, 0.5, n)
malignant = (rng.uniform(0, 1, n) < sigmoid(z)).astype(int)

scans = pd.DataFrame({"age": age, "lump_mm": lump_mm,
                      "family_history": family, "malignant": malignant})
print("malignant rate:", round(scans.malignant.mean(), 4))
scans.head()

malignant rate: 0.1378


,age,lump_mm,family_history,malignant
0,65.0,6.1,1.0,0
1,72.0,25.6,0.0,1
2,60.0,6.4,0.0,0
3,62.0,4.5,0.0,0
4,62.0,5.8,0.0,1


### B1 · "Our model is 90% accurate"

In [8]:
X = scans[["age", "lump_mm", "family_history"]]
y = scans["malignant"]

X_train, X_hidden, y_train, y_hidden = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y)

model = LogisticRegression(max_iter=2000).fit(X_train, y_train)

print(f"accuracy: {model.score(X_hidden, y_hidden):.4f}")
print()
print("REPORT: 'The screening model is 90% accurate. Recommend deployment.'")

accuracy: 0.9037

REPORT: 'The screening model is 90% accurate. Recommend deployment.'


**The problem:** accuracy is quoted with **no baseline**, on data where one class is rare.

Only about 14% of these scans are malignant — so a "model" that says *benign* to every
single patient is already right about 86% of the time, with no model at all.

90% against 86% is a gain of four points, not a triumph. And on a cancer screen, accuracy
is the wrong number entirely: what matters is **how many real cancers it caught**.

**The fix:** report the baseline and the confusion matrix.

In [9]:
baseline = max(y_hidden.mean(), 1 - y_hidden.mean())
pred = model.predict(X_hidden)
tn, fp, fn, tp = confusion_matrix(y_hidden, pred).ravel()

print(f"the model          : {model.score(X_hidden, y_hidden):.4f}")
print(f"always say 'benign': {baseline:.4f}")
print(f"the gain           : {model.score(X_hidden, y_hidden) - baseline:+.4f}")
print()
print(f"real cancers in the hidden set : {int(y_hidden.sum())}")
print(f"   the model flagged           : {tp}")
print(f"   the model MISSED            : {fn}")
print(f"   recall                      : {tp/(tp+fn):.4f}")

the model          : 0.9037
always say 'benign': 0.8630
the gain           : +0.0407

real cancers in the hidden set : 37
   the model flagged           : 16
   the model MISSED            : 21
   recall                      : 0.4324


```
the model          : 0.9037
always say 'benign': 0.8630
the gain           : +0.0407

real cancers in the hidden set : 37
   the model flagged           : 16
   the model MISSED            : 21
   recall                      : 0.4324
```

**The model misses more than half the cancers.**

It is 90% accurate because it is very good at the easy part — recognising the 233 benign
scans. On the 37 that matter it catches 16 and lets 21 through.

**"90% accurate, recommend deployment" would have put this into a hospital.** The
confusion matrix takes one line and would have stopped it.

### B2 · The cut-off nobody chose

In [10]:
pred = model.predict(X_hidden)              # <-- look here
tn, fp, fn, tp = confusion_matrix(y_hidden, pred).ravel()

print(f"we flag {tp + fp} patients for a biopsy")
print(f"we catch {tp} of the {tp + fn} real cancers")

we flag 21 patients for a biopsy
we catch 16 of the 37 real cancers


**The problem:** `.predict()` silently applies a **0.5 cut-off** that nobody chose. It
flags 21 patients and catches 16 of the 37 real cancers.

For a cancer screen that is close to the worst possible choice. A false alarm costs one
biopsy. A miss costs a life. Those are nothing like equal, so the cut-off should not be
the one you get when they are.

**The fix:** get the probabilities with `.predict_proba()` and choose the cut-off from the
cost of each mistake.

In [11]:
probs = model.predict_proba(X_hidden)[:, 1]

print(f"{'cut-off':>9}{'flagged':>10}{'caught':>9}{'missed':>9}{'recall':>9}{'accuracy':>11}")
print("-" * 57)
for t in [0.05, 0.10, 0.15, 0.25, 0.50]:
    p2 = (probs >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_hidden, p2).ravel()
    print(f"{t:>9}{tp+fp:>10}{tp:>9}{fn:>9}{tp/(tp+fn):>9.4f}{(p2 == y_hidden).mean():>11.4f}")

  cut-off   flagged   caught   missed   recall   accuracy
---------------------------------------------------------
     0.05       154       36        1   0.9730     0.5593
      0.1       104       30        7   0.8108     0.7000
     0.15        70       24       13   0.6486     0.7815
     0.25        40       20       17   0.5405     0.8630
      0.5        21       16       21   0.4324     0.9037


```
  cut-off   flagged   caught   missed   recall   accuracy
---------------------------------------------------------
     0.05       154       36        1   0.9730     0.5593
      0.1       104       30        7   0.8108     0.7000
     0.15        70       24       13   0.6486     0.7815
     0.25        40       20       17   0.5405     0.8630
      0.5        21       16       21   0.4324     0.9037
```

**Read the two end rows against each other.**

At **0.5** the model is most "accurate" (0.90) and misses **21 of 37** cancers.

At **0.05** accuracy drops to 0.56 — and it misses **1**. The cost is 154 biopsies instead
of 21.

**Which is right?** For a screening test, almost certainly something near 0.05–0.10. A
hundred and thirty extra biopsies against twenty lives is not a difficult trade.

And notice that **the most accurate model is the worst one here**. Accuracy and usefulness
are pulling in opposite directions, and only one of them matters.

### B3 · "Family history is the biggest risk factor"

In [12]:
for name, c in zip(X.columns, model.coef_[0]):
    print(f"{name:<18} {c:>+8.4f}")
print()
print("REPORT: 'Family history is by far the strongest predictor.'")

age                 +0.0468
lump_mm             +0.2933
family_history      +0.6096

REPORT: 'Family history is by far the strongest predictor.'


**The problem:** raw coefficients are being compared across columns with completely
different units — the same mistake as Day 34, and A4 above.

`family_history` is **0 or 1**. `age` runs from 20 to 90. `lump_mm` runs from 0 to 40.
A coefficient of 0.85 "per unit" of a 0/1 column is not comparable with 0.028 "per unit"
of a column that spans 70 units.

**The fix:** scale the columns, then compare.

In [13]:
scaler = StandardScaler().fit(X_train)
scaled = LogisticRegression(max_iter=2000).fit(scaler.transform(X_train), y_train)

out = pd.DataFrame({
    "column": X.columns,
    "raw": model.coef_[0].round(4),
    "spread (sd)": X.std().values.round(2),
    "scaled": scaled.coef_[0].round(4),
})
print(out.to_string(index=False))

        column    raw  spread (sd)  scaled
           age 0.0468        13.77  0.6450
       lump_mm 0.2933         4.37  1.1998
family_history 0.6096         0.41  0.2659


```
        column     raw  spread (sd)  scaled
           age  0.0468        13.77  0.6450
       lump_mm  0.2933         4.37  1.1998
family_history  0.6096         0.41  0.2659
```

**`lump_mm` is the strongest predictor by a factor of nearly two over `age`, and more than
four over `family_history`** — and it had the middle raw coefficient.

`family_history` goes from looking **first** to being **last**. Its raw coefficient was
largest only because the column can only ever move by 1, so "a one unit change" is the
entire column — there is nowhere else for it to go.

`age` moves the other way: the smallest raw coefficient by a long distance (0.047), and
**second** once you account for the fact that age spans 70 years.

**The report had it exactly backwards.** The thing a radiologist would have told you to
look at first — the size of the lump — is the thing the model agrees matters most, and the
raw coefficients hid it.

### B4 · The split that threw away the cases

In [14]:
big = scans[scans.lump_mm > 14]             # 52 patients with a large lump
print("rows:", len(big), "  malignant:", int(big.malignant.sum()))

a, b, c, d = train_test_split(big[["age", "lump_mm", "family_history"]],
                              big.malignant, test_size=0.3, random_state=7)

m = LogisticRegression(max_iter=2000).fit(a, c)
print("accuracy on the hidden rows:", round(m.score(b, d), 4))

rows: 52   malignant: 33
accuracy on the hidden rows: 0.625


**The problem:** no `stratify=` on a split of only **52 rows**.

The split is purely random, so the proportion of malignant cases in the hidden half is
left to chance. With 16 hidden rows that proportion swings wildly — and the accuracy
swings with it, for reasons that have nothing to do with the model.

**The fix:** `stratify=big.malignant` — stratify on the labels being split.

In [15]:
print(f"{'seed':>6}{'malignant in hidden half':>28}{'accuracy':>11}")
print("-" * 45)
for seed in range(6):
    a, b, c, d = train_test_split(big[["age", "lump_mm", "family_history"]],
                                  big.malignant, test_size=0.3, random_state=seed)
    m = LogisticRegression(max_iter=2000).fit(a, c)
    print(f"{seed:>6}{d.mean():>28.4f}{m.score(b, d):>11.4f}")

print()
print("and now with stratify:")
print(f"{'seed':>6}{'malignant in hidden half':>28}{'accuracy':>11}")
print("-" * 45)
for seed in range(6):
    a, b, c, d = train_test_split(big[["age", "lump_mm", "family_history"]],
                                  big.malignant, test_size=0.3, random_state=seed,
                                  stratify=big.malignant)
    m = LogisticRegression(max_iter=2000).fit(a, c)
    print(f"{seed:>6}{d.mean():>28.4f}{m.score(b, d):>11.4f}")

  seed    malignant in hidden half   accuracy
---------------------------------------------
     0                      0.5000     0.6250
     1                      0.5000     0.6875


     2                      0.5000     0.5625
     3                      0.5625     0.6250
     4                      0.7500     0.8750


     5                      0.5000     0.6250

and now with stratify:
  seed    malignant in hidden half   accuracy
---------------------------------------------
     0                      0.6250     0.7500
     1                      0.6250     0.8125
     2                      0.6250     0.6875
     3                      0.6250     0.5625


     4                      0.6250     0.7500
     5                      0.6250     0.6250


```
  seed    malignant in hidden half   accuracy
---------------------------------------------
     0                      0.5000     0.6250
     1                      0.5000     0.6875
     2                      0.5000     0.5625
     3                      0.5625     0.6250
     4                      0.7500     0.8750
     5                      0.5000     0.6250

and now with stratify:
  seed    malignant in hidden half   accuracy
---------------------------------------------
     0                      0.6250     0.7500
     1                      0.6250     0.8125
     2                      0.6250     0.6875
     3                      0.6250     0.5625
     4                      0.6250     0.7500
     5                      0.6250     0.6250
```

**Without stratify the hidden half's malignant rate wanders from 0.50 to 0.75** — a swing
of 25 points, purely from which rows happened to land where. Seed 4 reports **87.5%
accuracy**, and the only reason is that it happened to receive an easier hidden half.

With stratify, every split is 0.6250, matching the subgroup exactly.

### What to notice, honestly

The **accuracy still moves** with stratify on — 0.5625 to 0.8125. Stratifying fixes the
*class mix*; it does not make a single split on **16 hidden rows** a reliable measurement.

That needs **cross-validation**, which is tomorrow. Stratify is a necessary habit, not a
sufficient one — and if you ever see a result quoted from one split of a small dataset,
ask what the other splits said.

---
# Part C — Write the code

Five tasks on one dataset.

In [16]:
# ---------------------------------------------------------------
# 1,000 telecom customers. C1 to C5 all use `telco`.
# ---------------------------------------------------------------
rng = np.random.default_rng(3520)
N = 1000

tenure_months = rng.uniform(1, 72, N).round(0)
monthly_bill  = rng.normal(68, 22, N).clip(18, 130).round(2)
support_calls = rng.poisson(1.1, N).clip(0, 8).astype(float)
contract      = rng.choice(["monthly", "one_year", "two_year"], N, p=[.55, .28, .17])

bump = np.where(contract == "monthly", 1.15, np.where(contract == "one_year", 0.0, -0.95))
z = (-1.25 - 0.040*tenure_months + 0.021*monthly_bill + 0.38*support_calls + bump
     + rng.normal(0, 0.4, N))
churned = (rng.uniform(0, 1, N) < sigmoid(z)).astype(int)

telco = pd.DataFrame({"tenure_months": tenure_months, "monthly_bill": monthly_bill,
                      "support_calls": support_calls, "contract": contract,
                      "churned": churned})
telco.head()

,tenure_months,monthly_bill,support_calls,contract,churned
0,24.0,50.32,2.0,one_year,0
1,27.0,68.05,0.0,two_year,0
2,50.0,53.90,3.0,one_year,0
3,43.0,76.91,2.0,monthly,0
4,72.0,78.37,0.0,one_year,0


### C1 · Count before you fit

Step 1 for a classification problem is not a scatter plot — it is a **count**.

How many churned and how many did not? What is the baseline you have to beat? And does
the churn rate differ by contract type?

In [17]:
# your code here

In [18]:
print(telco.churned.value_counts().to_string())
print(f"\nchurn rate: {telco.churned.mean():.4f}")
print(f"baseline to beat (always guess the majority): "
      f"{max(telco.churned.mean(), 1 - telco.churned.mean()):.4f}")
print()
print(telco.groupby("contract").churned.agg(["size", "mean"]).round(3).to_string())

churned
0    580
1    420

churn rate: 0.4200
baseline to beat (always guess the majority): 0.5800



          size   mean
contract             
monthly    567  0.533
one_year   269  0.349
two_year   164  0.146


```
churned
0    580
1    420

churn rate: 0.4200
baseline to beat (always guess the majority): 0.5800

          size   mean
contract
monthly    567  0.533
one_year   269  0.349
two_year   164  0.146
```

**The baseline is 0.58** — say "will not churn" to everybody and you are right 58% of the
time. Any accuracy we report has to be read against that.

The classes are reasonably balanced (42/58), which is comfortable. If churn had been 3%
this would be a very different problem.

And the contract breakdown already tells a story before any model: **monthly customers
churn at 53%, two-year customers at 15%.** That is a 3.6× difference, and we should expect
the model to find it.

### C2 · Fit it and read it in odds

One-hot the contract column, split with `stratify`, fit, and print each coefficient
**together with what it does to the odds**. Report accuracy on both halves, and the log
loss on the hidden rows.

In [19]:
# your code here

In [20]:
numeric = ["tenure_months", "monthly_bill", "support_calls"]
encoded = pd.get_dummies(telco[numeric + ["contract"]],
                         columns=["contract"], drop_first=True).astype(float)
print("columns:", list(encoded.columns))

X_train, X_hidden, y_train, y_hidden = train_test_split(
    encoded, telco.churned, test_size=0.3, random_state=0, stratify=telco.churned)

model = LogisticRegression(max_iter=2000).fit(X_train, y_train)

print()
for name, c in zip(encoded.columns, model.coef_[0]):
    print(f"   {name:<22} {c:>+8.4f}   odds × {np.exp(c):.3f}")
print(f"   {'intercept':<22} {model.intercept_[0]:>+8.4f}")
print()
print(f"accuracy on rows it studied  : {model.score(X_train, y_train):.4f}")
print(f"accuracy on rows it never saw: {model.score(X_hidden, y_hidden):.4f}")
print(f"log loss on hidden rows      : {log_loss(y_hidden, model.predict_proba(X_hidden)):.4f}")

columns: ['tenure_months', 'monthly_bill', 'support_calls', 'contract_one_year', 'contract_two_year']



   tenure_months           -0.0404   odds × 0.960
   monthly_bill            +0.0201   odds × 1.020
   support_calls           +0.2169   odds × 1.242
   contract_one_year       -0.7428   odds × 0.476
   contract_two_year       -1.8680   odds × 0.154
   intercept               -0.0429

accuracy on rows it studied  : 0.7157
accuracy on rows it never saw: 0.6967
log loss on hidden rows      : 0.5428


```
columns: ['tenure_months', 'monthly_bill', 'support_calls',
          'contract_one_year', 'contract_two_year']

   tenure_months           -0.0404   odds × 0.960
   monthly_bill            +0.0201   odds × 1.020
   support_calls           +0.2169   odds × 1.242
   contract_one_year       -0.7428   odds × 0.476
   contract_two_year       -1.8680   odds × 0.154
   intercept               -0.0429

accuracy on rows it studied  : 0.7157
accuracy on rows it never saw: 0.6967
log loss on hidden rows      : 0.5428
```

**In English, holding everything else constant:**

- each extra **month** with us multiplies the churn odds by **0.96** — a 4% reduction per
  month, which compounds: a year of tenure is 0.96¹² ≈ **0.61**
- each **support call** multiplies the churn odds by **1.24**
- a **one-year** contract has about **half** the churn odds of a monthly one (the dropped
  baseline); a **two-year** contract has about **one sixth**

`contract_two_year` at ×0.154 is the single strongest effect in the model, and it matches
what the raw counts hinted at in C1.

### C3 · Which column matters most?

Rank the five columns properly. Show raw and scaled side by side, and say which column
the raw numbers would have misled you about.

In [21]:
# your code here

In [22]:
scaler = StandardScaler().fit(X_train)
scaled = LogisticRegression(max_iter=2000).fit(scaler.transform(X_train), y_train)

table = pd.DataFrame({
    "column": encoded.columns,
    "raw": model.coef_[0].round(4),
    "scaled": scaled.coef_[0].round(4),
})
table["rank by raw"] = table.raw.abs().rank(ascending=False).astype(int)
table["rank by scaled"] = table.scaled.abs().rank(ascending=False).astype(int)
print(table.sort_values("rank by scaled").to_string(index=False))

           column     raw  scaled  rank by raw  rank by scaled
    tenure_months -0.0404 -0.8688            4               1
contract_two_year -1.8680 -0.7585            1               2
     monthly_bill  0.0201  0.4368            5               3
contract_one_year -0.7428 -0.3414            2               4
    support_calls  0.2169  0.2213            3               5


```
           column     raw  scaled  rank by raw  rank by scaled
    tenure_months -0.0404 -0.8688            4               1
contract_two_year -1.8680 -0.7585            1               2
     monthly_bill  0.0201  0.4368            5               3
contract_one_year -0.7428 -0.3414            2               4
    support_calls  0.2169  0.2213            3               5
```

**`tenure_months` is the column the raw numbers would have ruined you on.** Its coefficient
is **−0.0404**, fourth of five and easy to dismiss — but tenure spans 1 to 72 months, so
across the real range it moves the odds more than anything else in the model.

Scaled, it is **first**.

And the mirror image: `support_calls` looks third by raw coefficient and is **last** once
scaled, because almost everybody makes 0, 1 or 2 calls. The column barely moves, so it
cannot do much work however large its per-unit effect.

**Raw coefficients rank by units. Scaled coefficients rank by influence.**

### C4 · Choose a cut-off, and defend it

The retention team can call **at most 120** of the 300 hidden customers, and an offer
costs ₹600. A customer who churns is worth ₹9,000 in lost lifetime value.

Build the threshold table, then pick a cut-off and justify it **with the numbers**.

In [23]:
# your code here

In [24]:
probs = model.predict_proba(X_hidden)[:, 1]

print(f"{'cut':>6}{'we call':>9}{'caught':>8}{'missed':>8}{'recall':>9}"
      f"{'accuracy':>10}{'cost':>12}{'saved':>12}{'net':>12}")
print("-" * 86)
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    pred = (probs >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_hidden, pred).ravel()
    called = tp + fp
    cost = called * 600
    saved = tp * 9000
    print(f"{t:>6}{called:>9}{tp:>8}{fn:>8}{tp/(tp+fn):>9.4f}"
          f"{(pred == y_hidden).mean():>10.4f}{cost:>12,}{saved:>12,}{saved-cost:>12,}")

   cut  we call  caught  missed   recall  accuracy        cost       saved         net
--------------------------------------------------------------------------------------
   0.3      195     112      14   0.8889    0.6767     117,000   1,008,000     891,000
   0.4      156      97      29   0.7698    0.7067      93,600     873,000     779,400
   0.5      115      75      51   0.5952    0.6967      69,000     675,000     606,000
   0.6       81      59      67   0.4683    0.7033      48,600     531,000     482,400
   0.7       54      40      86   0.3175    0.6667      32,400     360,000     327,600


```
   cut  we call  caught  missed   recall  accuracy        cost       saved         net
--------------------------------------------------------------------------------------
   0.3      195     112      14   0.8889    0.6767     117,000   1,008,000     891,000
   0.4      156      97      29   0.7698    0.7067      93,600     873,000     779,400
   0.5      115      75      51   0.5952    0.6967      69,000     675,000     606,000
   0.6       81      59      67   0.4683    0.7033      48,600     531,000     482,400
   0.7       54      40      86   0.3175    0.6667      32,400     360,000     327,600
```

### The answer: **0.5**, because of the capacity limit — not because it is the default.

The money alone says go as low as possible: **0.3 nets ₹891,000**, the best row in the
table. But it means calling **195 customers**, and the team can only call **120**.

Of the rows that fit the constraint, **0.5 calls 115 and nets ₹606,000** — the best
feasible option.

### Three things worth saying about this

**The most accurate cut-off is not the most profitable one.** 0.4 has the highest accuracy
(0.7067) and nets ₹779,400. **0.3 is the second *worst* row for accuracy (0.6767) and the
best row for money.** Accuracy was never the objective — rupees were.

**The constraint is doing the real work.** The honest recommendation to management is not
"use 0.5". It is: *going from 115 calls to 195 earns another ₹285,000 — about **₹3,560 per
extra call made**. Can we have more callers?* At a ₹600 offer cost, almost any way of
adding capacity pays for itself several times over.

**A better rule than any threshold: sort by probability and work down the list** until the
callers run out. That captures the highest-risk customers regardless of where the cut-off
happens to fall.

### C5 · Score three real customers

For each, give the probability and say what you would do.

| | tenure | bill | support calls | contract |
|---|---|---|---|---|
| **A** | 3 months | ₹95 | 4 | monthly |
| **B** | 60 months | ₹55 | 0 | two_year |
| **C** | 18 months | ₹78 | 2 | one_year |

In [25]:
# your code here

In [26]:
people = pd.DataFrame([
    {"tenure_months": 3,  "monthly_bill": 95, "support_calls": 4,
     "contract_one_year": 0.0, "contract_two_year": 0.0},
    {"tenure_months": 60, "monthly_bill": 55, "support_calls": 0,
     "contract_one_year": 0.0, "contract_two_year": 1.0},
    {"tenure_months": 18, "monthly_bill": 78, "support_calls": 2,
     "contract_one_year": 1.0, "contract_two_year": 0.0},
])[encoded.columns]

p = model.predict_proba(people)[:, 1]
for label, prob in zip("ABC", p):
    print(f"customer {label}:  p(churn) = {prob:.4f}   odds = {prob/(1-prob):.3f}")

customer A:  p(churn) = 0.9319   odds = 13.691
customer B:  p(churn) = 0.0382   odds = 0.040
customer C:  p(churn) = 0.6206   odds = 1.636


In [27]:
# and show WHERE customer A's risk comes from
row = people.iloc[0]
total = model.intercept_[0]
print(f"{'intercept':<22}{'':>7}   {model.intercept_[0]:>+8.4f}")
for name, c in zip(encoded.columns, model.coef_[0]):
    piece = row[name] * c
    total += piece
    print(f"{name:<22}{row[name]:>7} × {c:>+8.4f} = {piece:>+8.4f}")
print("-" * 50)
print(f"{'z':<22}{'':>7}   {total:>+8.4f}")
print(f"{'p':<22}{'':>7}    {sigmoid(total):>7.4f}")

intercept                        -0.0429
tenure_months             3.0 ×  -0.0404 =  -0.1211
monthly_bill             95.0 ×  +0.0201 =  +1.9130
support_calls             4.0 ×  +0.2169 =  +0.8678
contract_one_year         0.0 ×  -0.7428 =  -0.0000
contract_two_year         0.0 ×  -1.8680 =  -0.0000
--------------------------------------------------
z                                +2.6168
p                                 0.9319


```
customer A:  p(churn) = 0.9319   odds = 13.691
customer B:  p(churn) = 0.0382   odds = 0.040
customer C:  p(churn) = 0.6206   odds = 1.636
```

**Customer A — call today.** A **93%** churn probability, and every factor points the same
way: three months in, a ₹95 bill, four support calls, no contract lock-in. The expected
value of the call is 0.93 × ₹9,000 ≈ **₹8,400** against a ₹600 offer.

**Customer B — do not call.** Under **4%**. Five years with us, a cheap plan, no
complaints, locked into a two-year contract. Spending ₹600 here is close to pure waste.

**Customer C — call, but she is not urgent.** At **0.62** she is above the 0.5 cut-off and
would be flagged, but her risk is less than a third of A's odds-wise (1.64 against 13.69).
If the team can only make a limited number of calls, A goes first.

### Where A's risk comes from

```
intercept                        -0.0429
tenure_months            3.0 ×  -0.0404 =  -0.1211
monthly_bill            95.0 ×  +0.0201 =  +1.9130
support_calls            4.0 ×  +0.2169 =  +0.8678
contract_one_year        0.0 ×  -0.7428 =  -0.0000
contract_two_year        0.0 ×  -1.8680 =  -0.0000
--------------------------------------------------
z                                +2.6168
p                                 0.9319
```

**The monthly bill is doing most of the damage** — ₹95 against a ₹68 average, and at
+0.0201 per rupee that single column contributes **+1.91** to z. The four support calls add
a further **+0.87**. Her three months of tenure barely subtract anything (−0.12), because
tenure's work happens over *years*: at 60 months it would have been −2.42, which is how
customer B gets to 4%.

**This is the real value of a probability model over a yes/no one.** The retention team
does not just learn *who* to call — they learn *why*, and can therefore make an offer that
addresses the actual problem. For A, that is the bill.

---
# Part D — Challenge

Two tasks. Both are marked.

### D1 · Build logistic regression from scratch

Write `fit_logistic(X, y, rate, steps)` using no scikit-learn. It should:

- work for **any number of columns**, not just one
- use the gradient descent loop from yesterday, with the sigmoid in it
- report its progress so a reader can watch the log loss fall
- handle the scale problem — `monthly_bill` is in the tens while `support_calls` is 0 to 8

Then prove it matches `LogisticRegression(C=1e9)` on the telecom data.

*Reminder of the loop:*
```
p     = sigmoid(X @ w + b)
error = p - y                      # same as always
w    -= rate × (1/n) × (X.T @ error)
b    -= rate × (1/n) × error.sum()
```

In [28]:
# the telecom data from Part C, one-hot encoded, as a plain numpy array
encoded = pd.get_dummies(
    telco[["tenure_months", "monthly_bill", "support_calls", "contract"]],
    columns=["contract"], drop_first=True).astype(float)

demo_X = encoded.values
demo_y = telco.churned.values
print(demo_X.shape, demo_y.shape, list(encoded.columns))

(1000, 5) (1000,) ['tenure_months', 'monthly_bill', 'support_calls', 'contract_one_year', 'contract_two_year']


In [29]:
# your function here

In [30]:
def my_log_loss(y, p):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return float(-(y*np.log(p) + (1-y)*np.log(1-p)).mean())


def fit_logistic(X, y, rate=0.5, steps=30000, report=(1, 10, 100, 1000, 10000, 30000)):
    """Yesterday's gradient descent loop, with a sigmoid in front of it.

    Scales inside the function (Day 32) because the columns have very
    different ranges, then converts the answer back to the caller's units.
    """
    X = np.asarray(X, float)
    y = np.asarray(y, float)
    n, k = X.shape

    centre, spread = X.mean(axis=0), X.std(axis=0)
    spread[spread == 0] = 1.0
    Xs = (X - centre) / spread

    w = np.zeros(k)
    b = 0.0
    for step in range(1, steps + 1):
        p = sigmoid(Xs @ w + b)
        error = p - y                                 # the whole gradient, in one line
        w -= rate * (1/n) * (Xs.T @ error)
        b -= rate * (1/n) * error.sum()
        if step in report:
            print(f"   step {step:>6}:  log loss {my_log_loss(y, p):.6f}")

    # undo the scaling so the coefficients mean something to the caller
    return w / spread, b - (w * centre / spread).sum()

In [31]:
print("gradient descent:")
my_w, my_b = fit_logistic(demo_X, demo_y)

theirs = LogisticRegression(C=1e9, max_iter=20000).fit(demo_X, demo_y)

print()
out = pd.DataFrame({
    "column": encoded.columns,
    "mine": my_w.round(5),
    "sklearn": theirs.coef_[0].round(5),
})
print(out.to_string(index=False))
print()
print(f"intercept   mine {my_b:.5f}   sklearn {theirs.intercept_[0]:.5f}")
print(f"log loss    mine {my_log_loss(demo_y, sigmoid(demo_X @ my_w + my_b)):.6f}"
      f"   sklearn {log_loss(demo_y, theirs.predict_proba(demo_X)):.6f}")

gradient descent:
   step      1:  log loss 0.693147
   step     10:  log loss 0.573305
   step    100:  log loss 0.544537


   step   1000:  log loss 0.544535


   step  10000:  log loss 0.544535


   step  30000:  log loss 0.544535

           column     mine  sklearn
    tenure_months -0.04012 -0.04012
     monthly_bill  0.02169  0.02169
    support_calls  0.23258  0.23258
contract_one_year -0.93168 -0.93167
contract_two_year -2.14309 -2.14309

intercept   mine -0.10760   sklearn -0.10760
log loss    mine 0.544535   sklearn 0.544535


In [32]:
# prove it is general: a different dataset, 6 columns, wildly different scales
gen = np.random.default_rng(606)
Xg = gen.normal(0, 1, (900, 6)) * gen.uniform(0.5, 300, 6)
true_w = np.array([0.9, -0.4, 0.0, 1.7, -1.1, 0.3]) / gen.uniform(0.5, 300, 6)
yg = (gen.uniform(0, 1, 900) < sigmoid(Xg @ true_w + 0.3)).astype(float)

mine_w, mine_b = fit_logistic(Xg, yg, rate=0.5, steps=30000, report=())
sk = LogisticRegression(C=1e9, max_iter=20000).fit(Xg, yg)

print("the same data, default C=1 vs C=1e9:")
soft = LogisticRegression(C=1, max_iter=20000).fit(demo_X, demo_y)
print(pd.DataFrame({"column": encoded.columns,
                    "C=1": soft.coef_[0].round(4),
                    "C=1e9": theirs.coef_[0].round(4),
                    "shrunk?": np.abs(soft.coef_[0]) < np.abs(theirs.coef_[0])}
                   ).to_string(index=False))
print()
print("six columns, scales from 0.5 to 300:")
print(f"   biggest coefficient difference: {np.abs(mine_w - sk.coef_[0]).max():.2e}")
print(f"   intercept difference          : {abs(mine_b - sk.intercept_[0]):.2e}")
print(f"   log loss  mine {my_log_loss(yg, sigmoid(Xg @ mine_w + mine_b)):.6f}"
      f"   sklearn {log_loss(yg, sk.predict_proba(Xg)):.6f}")

the same data, default C=1 vs C=1e9:
           column     C=1   C=1e9  shrunk?
    tenure_months -0.0398 -0.0401     True
     monthly_bill  0.0214  0.0217     True
    support_calls  0.2310  0.2326     True
contract_one_year -0.8861 -0.9317     True
contract_two_year -2.0092 -2.1431     True

six columns, scales from 0.5 to 300:
   biggest coefficient difference: 6.92e-08
   intercept difference          : 6.26e-06
   log loss  mine 0.217083   sklearn 0.217083


### The result

```
gradient descent:
   step      1:  log loss 0.693147
   step     10:  log loss 0.573305
   step    100:  log loss 0.544537
   step   1000:  log loss 0.544535
   step  10000:  log loss 0.544535
   step  30000:  log loss 0.544535

           column     mine  sklearn
    tenure_months -0.04012 -0.04012
     monthly_bill  0.02169  0.02169
    support_calls  0.23258  0.23258
contract_one_year -0.93168 -0.93167
contract_two_year -2.14309 -2.14309

intercept   mine -0.10760   sklearn -0.10760
log loss    mine 0.544535   sklearn 0.544535

six columns, scales from 0.5 to 300:
   biggest coefficient difference: 6.92e-08
   intercept difference          : 6.26e-06
   log loss  mine 0.217083   sklearn 0.217083
```

**Identical to four decimal places or better** — the one visible disagreement is
`contract_one_year` in the fifth decimal — and the log loss agrees to all six.

Notice too how fast it gets there: the log loss is already 0.5445 by **step 100**, and the
remaining 29,900 steps buy an improvement in the fourth decimal place. Gradient descent
does most of its work early.

### The three things that earn the marks

**1. `error = p - y`.** This is the whole gradient, and it is the *same expression* you
used for linear regression yesterday — the only change is that `p` now comes out of a
sigmoid. That is not a coincidence: the derivative of log loss with respect to z works out
to exactly prediction minus truth, which is the reason log loss is the natural partner for
the sigmoid.

**2. Scaling inside the function.** `monthly_bill` runs to 130 while `support_calls` stops
at 8. Without scaling, any learning rate small enough to keep the bill stable leaves the
call count crawling. Scaling inside, and converting back at the end, means the caller
never has to know.

**3. Comparing against `C=1e9`, not the default.** The default regularises, so it would
*not* have matched — and a student who compares against it will think their loop is
broken when it is correct.

Fit the same full dataset with the default `C=1` and compare:

| | default `C=1` | `C=1e9` |
|---|---|---|
| `tenure_months` | −0.0398 | −0.0401 |
| `monthly_bill` | +0.0214 | +0.0217 |
| `support_calls` | +0.2310 | +0.2326 |
| `contract_one_year` | −0.8861 | **−0.9317** |
| `contract_two_year` | −2.0092 | **−2.1431** |

**Every single coefficient is smaller under the default**, and the two large ones are
pulled in hardest — `contract_two_year` by 0.13. That is scikit-learn quietly applying
**regularisation** you never asked for: a penalty on large coefficients, added to the cost
function. It is usually a good thing, which is why it is the default.

It is also the subject of a day of its own later in this module. For now the point is
narrower: **know which model you are comparing against before you conclude your code is
broken.**

### What it tells you about yesterday

The loop you wrote for linear regression needed **one change** to become a classifier:
wrap the prediction in a sigmoid. Everything else — the error, the two update lines, the
learning rate, the scaling — is unchanged.

That is why gradient descent was worth a day of its own.

### D2 · A screener that auto-rejects job applicants

A company has deployed the model below to filter applications. It reports:

> *"The screening model is **86% accurate** on applications it has never seen. Candidates
> the model scores below 0.5 are auto-rejected without human review. This saves the
> recruiting team roughly 90% of their reading time."*

The model is correctly fitted and correctly split. The accuracy figure is true.

**Your tasks:**

1. Run it and confirm the 86%.
2. Find **three separate problems** with deploying it as described.
3. Prove each one with numbers.
4. Write a short recommendation — not "don't use it", but what you would change.

In [33]:
rng = np.random.default_rng(3530)
A_ = 1200

years_experience   = rng.uniform(0, 12, A_).round(1)
skill_test         = rng.normal(58, 16, A_).clip(10, 100).round(0)
referral           = rng.binomial(1, 0.18, A_).astype(float)
portfolio_projects = rng.poisson(1.6, A_).clip(0, 8).astype(float)

z = (-6.5 + 0.16*years_experience + 0.052*skill_test + 1.35*referral
     + 0.22*portfolio_projects + rng.normal(0, 0.55, A_))
hired = (rng.uniform(0, 1, A_) < sigmoid(z)).astype(int)

apps = pd.DataFrame({"years_experience": years_experience, "skill_test": skill_test,
                     "referral": referral, "portfolio_projects": portfolio_projects,
                     "hired": hired})
apps.head()

,years_experience,skill_test,referral,portfolio_projects,hired
0,8.1,94.0,0.0,0.0,1
1,0.5,51.0,0.0,4.0,0
2,6.6,76.0,1.0,1.0,0
3,0.7,56.0,1.0,2.0,0
4,5.7,52.0,0.0,5.0,1


In [34]:
feat = ["years_experience", "skill_test", "referral", "portfolio_projects"]

a, b, c, d = train_test_split(apps[feat], apps.hired, test_size=0.3,
                              random_state=0, stratify=apps.hired)
screener = LogisticRegression(max_iter=2000).fit(a, c)

print(f"accuracy on applications it has never seen: {screener.score(b, d):.4f}")

accuracy on applications it has never seen: 0.8611


In [35]:
# your investigation here

In [36]:
# ---- PROBLEM 1 -------------------------------------------------------
print("what does doing nothing score?")
print(f"   hire rate overall        : {apps.hired.mean():.4f}")
print(f"   'auto-reject everyone'   : {1 - d.mean():.4f}")
print(f"   the model                : {screener.score(b, d):.4f}")
print(f"   the gain                 : {screener.score(b, d) - (1 - d.mean()):+.4f}")

what does doing nothing score?
   hire rate overall        : 0.1842
   'auto-reject everyone'   : 0.8167
   the model                : 0.8611
   the gain                 : +0.0444


In [37]:
# ---- PROBLEM 2 -------------------------------------------------------
pred = screener.predict(b)
tn, fp, fn, tp = confusion_matrix(d, pred).ravel()
print(f"of the {int(d.sum())} applicants who WOULD have been hired:")
print(f"   the model passed through : {tp}")
print(f"   the model auto-rejected  : {fn}")
print(f"   recall                   : {tp/(tp+fn):.4f}")
print()
print(f"the team reads {tp+fp} applications instead of {len(d)} "
      f"— a {1-(tp+fp)/len(d):.0%} saving")

of the 66 applicants who WOULD have been hired:
   the model passed through : 22
   the model auto-rejected  : 44
   recall                   : 0.3333

the team reads 28 applications instead of 360 — a 92% saving


In [38]:
# ---- what a lower cut-off would buy ----------------------------------
probs = screener.predict_proba(b)[:, 1]
print(f"{'cut':>6}{'read':>7}{'caught':>8}{'rejected in error':>20}{'recall':>9}"
      f"{'reading saved':>15}")
print("-" * 66)
for t in [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]:
    p2 = (probs >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(d, p2).ravel()
    print(f"{t:>6}{tp+fp:>7}{tp:>8}{fn:>20}{tp/(tp+fn):>9.4f}"
          f"{1-(tp+fp)/len(d):>14.0%}")

   cut   read  caught   rejected in error   recall  reading saved
------------------------------------------------------------------
  0.05    295      65                   1   0.9848           18%
   0.1    222      63                   3   0.9545           38%
  0.15    170      58                   8   0.8788           53%
   0.2    126      54                  12   0.8182           65%
   0.3     73      41                  25   0.6212           80%
   0.5     28      22                  44   0.3333           92%


In [39]:
# ---- PROBLEM 3 -------------------------------------------------------
print("the coefficients, in odds:")
for name, co in zip(feat, screener.coef_[0]):
    print(f"   {name:<22} {co:>+8.4f}   odds × {np.exp(co):.3f}")
print()
print(f"applicants with a referral: {apps.referral.mean():.1%}")
print(f"   hire rate WITH one   : {apps[apps.referral == 1].hired.mean():.4f}")
print(f"   hire rate WITHOUT one: {apps[apps.referral == 0].hired.mean():.4f}")

the coefficients, in odds:
   years_experience        +0.1355   odds × 1.145
   skill_test              +0.0642   odds × 1.066
   referral                +1.1579   odds × 3.183
   portfolio_projects      +0.2852   odds × 1.330

applicants with a referral: 17.9%
   hire rate WITH one   : 0.3535
   hire rate WITHOUT one: 0.1472


### Problem 1 — the 86% is barely better than doing nothing

```
hire rate overall        : 0.1842
'auto-reject everyone'   : 0.8167
the model                : 0.8611
the gain                 : +0.0444
```

Only **18% of applicants are hired**, so a system that auto-rejects *every single
application* is **81.7% accurate**. The model's 86.1% is a gain of **four percentage
points** over a machine that does nothing at all.

"86% accurate" is a true statement that creates a completely false impression, and it is
the only number in the report.

### Problem 2 — it auto-rejects two thirds of the people it should hire

```
of the 66 applicants who WOULD have been hired:
   the model passed through : 22
   the model auto-rejected  : 44
   recall                   : 0.3333

the team reads 28 applications instead of 360 — a 92% saving
```

**Forty-four people who would have got the job were rejected without a human ever
reading their application.** The model catches **one in three**.

The 90% time saving in the report is real. It is achieved by throwing away most of the
good candidates along with the bad ones.

### What a lower cut-off would buy

```
   cut   read  caught   rejected in error   recall  reading saved
------------------------------------------------------------------
  0.05    295      65                   1   0.9848             18%
   0.1    222      63                   3   0.9545             38%
  0.15    170      58                   8   0.8788             53%
   0.2    126      54                  12   0.8182             65%
   0.3     73      41                  25   0.6212             80%
   0.5     28      22                  44   0.3333             92%
```

**At 0.2 the team reads 126 applications instead of 28 — and loses 12 good candidates
instead of 44.** That is still a 65% saving on reading time, for a third of the damage.

The 0.5 cut-off was never chosen. It is simply what `.predict()` does.

### Problem 3 — the biggest coefficient is the one with nothing to do with ability

```
   years_experience        +0.1355   odds × 1.145
   skill_test              +0.0642   odds × 1.066
   referral                +1.1579   odds × 3.183
   portfolio_projects      +0.2852   odds × 1.330

applicants with a referral: 17.9%
   hire rate WITH one   : 0.3535
   hire rate WITHOUT one: 0.1472
```

**`referral` is by far the largest coefficient — it multiplies the odds by 3.18.**

The model is not wrong: referred candidates genuinely were hired more often in this data.
But a referral measures **who you already know**, not what you can do. Bake that into an
automatic rejection and you have built a system that systematically filters out everyone
without a contact at the company — and it will keep doing so, invisibly, at scale.

*(Note too that `referral` only moves between 0 and 1. On a scaled comparison it would
rank lower. The point is not that it dominates the model arithmetically — it is that it is
in there at all, in a system that rejects people automatically.)*

### The recommendation

> The 86% figure is not meaningful on its own: auto-rejecting every application scores
> 81.7%, so the model adds four points. More importantly, at the current 0.5 cut-off it
> auto-rejects **44 of the 66 candidates who would have been hired** — it catches one in
> three. Moving the cut-off to **0.2** cuts that to 12 missed candidates while still saving
> the team 65% of their reading. I would also **remove `referral` from a model that
> rejects automatically**: it is the largest single factor and it measures professional
> network rather than capability. And I would not auto-reject at all — use the score to
> **order** the reading queue, so that every application is still seen by a person, just in
> a better order.

### The idea to take away

Every one of these three problems is a **today** problem:

1. **accuracy without a baseline** — section 5
2. **a cut-off nobody chose** — section 4
3. **reading a coefficient without asking what it measures** — section 4, and Day 34

None of them needed a more advanced technique to spot. They needed somebody to ask the
three questions at the top of this notebook.

---
## Before you hand it in

- Did you run every coefficient through `exp()` before describing it?
- Did you report accuracy **next to a baseline**?
- Did you use `.predict_proba()` rather than `.predict()`?
- Did you say **who chose the cut-off**, and what each mistake costs?
- Did you add `stratify=y` to the split?
- For every claim: does it carry "holding the other columns constant"?

Four numbers worth carrying away:

- A cancer screen at **90% accuracy** caught **16 of 37** tumours. At a cut-off of 0.05 it
  caught 36 — and its accuracy fell to 56%.
- A hiring screener at **86% accuracy** beat "reject everyone" by **4.4 points**, and
  auto-rejected **44 of 66** people who would have been hired.
- `tenure_months` had the **smallest** raw coefficient in the churn model (−0.040) and the
  **largest** influence once scaled.
- Our own gradient descent loop and `LogisticRegression(C=1e9)` agreed to **four decimal
  places** — the loop from yesterday, with one sigmoid added.

Answers will be shared after class.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*